In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
## On enlève les lignes où y'a pas de titres et on remplace les chiffre à virgule par chiffre à point ex : 11,5 -> 11.5

scores = spark.table("films.bronze.scores")
scores = scores.filter(F.col("Titre").isNotNull())
scores = scores.withColumn("Unnamed: 11",F.replace(F.col("Unnamed: 11"), F.lit(","), F.lit(".")))

display(scores)

In [0]:
##On garde les colonnes qu'on veut

clean_scores = scores.select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("Date"),
    F.col("Titre").try_cast("string").alias("Titre"),
    F.col("Note Mathias").try_cast("double").alias("Note Cinématographie Mathias"),
    F.col("Unnamed: 7").try_cast("double").alias("Note Scénario Mathias"),
    F.col("Unnamed: 8").try_cast("double").alias("Note Appréciation générale Mathias"),
    F.col("Note Zélie").try_cast("double").alias("Note Cinématographie Zélie"),
    F.col("Unnamed: 10").try_cast("double").alias("Note Scénario Zélie"),
    F.col("Unnamed: 11").try_cast("double").alias("Note Appréciation générale Zélie")
)

In [0]:
## Création d'une colonne Note totale qui est une pondération arbitraire des 3 autres notes

clean_scores = clean_scores.withColumn("Note Totale Mathias", 
                                        F.coalesce(
                                            F.round(
                                                F.col("Note Cinématographie Mathias")*0.3 + F.col("Note Scénario Mathias")*0.3 + F.col("Note Appréciation générale Mathias")*0.4
                                                ,1
                                            ),
                                        F.col("Note Appréciation générale Mathias")))

clean_scores = clean_scores.withColumn("Note Totale Zélie", 
                                        F.coalesce(
                                            F.round(
                                                F.col("Note Cinématographie Zélie")*0.3 + F.col("Note Scénario Zélie")*0.3 + F.col("Note Appréciation générale Zélie")*0.4
                                                ,1
                                            ),
                                        F.col("Note Appréciation générale Zélie")))


In [0]:
clean_scores = clean_scores.withColumn("Titre", F.trim("Titre"))

In [0]:
### Dépivotage des notes

clean_scores = clean_scores.unpivot(["Date","Titre"], ["Note Cinématographie Mathias","Note Scénario Mathias","Note Appréciation générale Mathias","Note Cinématographie Zélie","Note Scénario Zélie","Note Appréciation générale Zélie","Note totale Mathias","Note totale Zélie"],"Type_note","Note")

In [0]:
clean_scores = clean_scores.withColumn("Nom", F.when(F.col("Type_note").contains("Mathias"),"Mathias").otherwise("Zélie"))   ### création d'une nouvelle colonne pour qui a voté
clean_scores = clean_scores.withColumn("Type_note", F.regexp_replace(F.col("Type_note"),"Note|Mathias|Zélie",""))                      ### suppression du nom dans le type
clean_scores = clean_scores.filter(F.col("Note").isNotNull())

# Ecriture dans le silver

In [0]:
clean_scores.write.mode("overwrite").option("overwriteSchema",True).saveAsTable("films.silver.clean_scores")